<a href="https://colab.research.google.com/github/jkj05/BIS5/blob/main/CuckooSearch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:

import random
import math

sensors = [(1, 1), (3, 1), (5, 1), (1, 4), (3, 4), (5, 4)]
sites = [(x, y) for y in (0, 2.5, 5) for x in (0, 2.5, 5)]

n = 25
Pa = 0.25
iterations = 250
required_power = 1.5
penalty = 100


def levy_flight():
    beta = 1.5
    sigma = (
        math.gamma(1 + beta) * math.sin(math.pi * beta / 2) /
        (math.gamma((1 + beta) / 2) * beta * 2 ** ((beta - 1) / 2))
    ) ** (1 / beta)

    u = random.gauss(0, sigma)
    v = random.gauss(0, 1)

    return 0.01 * u / abs(v) ** (1 / beta)


def charging_power(sensor, chargers):
    return sum(
        10 / (1 + (sensor[0] - x) ** 2 + (sensor[1] - y) ** 2)
        for x, y in chargers
    )


def fitness(nest):
    chargers = [p for p, value in zip(sites, nest) if value > 0.5]

    shortage = sum(
        max(0, required_power - charging_power(sensor, chargers)) ** 2
        for sensor in sensors
    )

    return len(chargers) + penalty * shortage


nests = [[random.random() for _ in sites] for _ in range(n)]
scores = [fitness(nest) for nest in nests]

best_index = min(range(n), key=lambda i: scores[i])
best = nests[best_index][:]
best_score = scores[best_index]

for _ in range(iterations):

    i = random.randrange(n)

    new_nest = [
        max(0, min(1, value + levy_flight()))
        for value in nests[i]
    ]

    new_score = fitness(new_nest)

    j = random.randrange(n)

    if new_score < scores[j]:
        nests[j] = new_nest
        scores[j] = new_score

    best_index = min(range(n), key=lambda i: scores[i])

    if scores[best_index] < best_score:
        best = nests[best_index][:]
        best_score = scores[best_index]

    worst = sorted(
        range(n),
        key=lambda i: scores[i],
        reverse=True
    )

    for j in worst:
        if j != best_index:
            if worst.index(j) < int(Pa * n):
                nests[j] = [random.random() for _ in sites]
                scores[j] = fitness(nests[j])

chargers = [p for p, value in zip(sites, best) if value > 0.5]

print("\nOPTIMAL WIRELESS CHARGER DEPLOYMENT")
print("-----------------------------------")
print("Number of chargers:", len(chargers))

for i, position in enumerate(chargers, 1):
    print(f"Charger {i}: {position}")

print("\nSENSOR CHARGING POWER")
print("-----------------------------------")

for i, sensor in enumerate(sensors, 1):
    power = charging_power(sensor, chargers)
    print(f"Sensor {i}: {power:.2f}")

print("\nFinal Fitness:", round(best_score, 4))



OPTIMAL WIRELESS CHARGER DEPLOYMENT
-----------------------------------
Number of chargers: 2
Charger 1: (2.5, 0)
Charger 2: (2.5, 5)

SENSOR CHARGING POWER
-----------------------------------
Sensor 1: 2.87
Sensor 2: 5.02
Sensor 3: 1.64
Sensor 4: 2.87
Sensor 5: 5.02
Sensor 6: 1.64

Final Fitness: 2
